# 09 — Recommendations


In [ ]:
import sys
from collections import defaultdict
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

from build_optimiser.config import Config
from build_optimiser.graph import attach_metrics, load_graph
from build_optimiser.simulation import rebuild_cost, simulate_merge

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Load All Results

Read recommendation prerequisites from NB04–NB08 outputs.


In [ ]:
results_dir = Path('../data/results')
results_dir.mkdir(parents=True, exist_ok=True)


def safe_read_parquet(path, fallback_cols=None):
    p = Path(path)
    if p.exists():
        try:
            return pd.read_parquet(p)
        except Exception:
            pass
    print(f'Warning: {p} not found or empty.')
    return pd.DataFrame(columns=fallback_cols or [])


def safe_read_csv(path, fallback_cols=None):
    p = Path(path)
    if p.exists():
        try:
            df = pd.read_csv(p)
            return df
        except Exception:
            pass
    print(f'Warning: {p} not found or empty.')
    return pd.DataFrame(columns=fallback_cols or [])


# ── Load upstream outputs ─────────────────────────────────────────────────────
split_recs = safe_read_parquet(
    '../data/results/split_recommendations.parquet',
    fallback_cols=['cmake_target', 'feature_group', 'file_count', 'compile_time_sum_ms',
                   'compile_saving_ms', 'recommended_k', 'method', 'balance_ratio',
                   'cross_partition_includes', 'gen_files', 'notes'],
)
optimised_mapping = safe_read_csv(
    '../data/results/optimised_feature_mapping.csv',
    fallback_cols=['executable', 'scenario', 'feature_groups_required'],
)
thin_df = safe_read_csv(
    '../data/results/thin_dependencies.csv',
    fallback_cols=['depending_target', 'depended_target', 'used_headers',
                   'total_headers', 'thinness_ratio'],
)
fg_df = safe_read_parquet(
    '../data/processed/feature_group_assignments.parquet',
    fallback_cols=['cmake_target', 'feature_group'],
)
ownership_df = safe_read_parquet(
    '../data/processed/target_ownership.parquet',
    fallback_cols=['cmake_target', 'group_id', 'ownership_normalised', 'owning_group_id'],
)

# Derive owning_group_label if not present
if not ownership_df.empty and 'owning_group_label' not in ownership_df.columns:
    groups_path = Path('../data/processed/contributor_groups.parquet')
    if groups_path.exists():
        groups_df = pd.read_parquet(groups_path)
        label_map = groups_df[['group_id', 'group_label']].drop_duplicates().set_index('group_id')['group_label']
        ownership_df['owning_group_label'] = ownership_df['owning_group_id'].map(label_map).fillna('unknown')

# Compute critical path set for on_cp annotations
from build_optimiser.graph import critical_path
cp_nodes = critical_path(G, weight_attr='total_build_time_ms')
from build_optimiser.graph import critical_path_length
cp_length = critical_path_length(G, weight_attr='total_build_time_ms')
cp_set = set(cp_nodes)

# Load risk ratings from nb08 if available
risk_df = safe_read_parquet(
    '../data/results/risk_analysis.parquet',
    fallback_cols=['cmake_target', 'risk_rating'],
)
risk_lookup = dict(zip(risk_df['cmake_target'], risk_df['risk_rating'])) if not risk_df.empty else {}

# Compute daily rebuild costs for scoring
working_days = max(getattr(cfg, 'git_history_months', 6) * 20, 1)
cost_rows = []
for _, row in target_df.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    commit_count = row.get('git_commit_count_total', 0) or 0
    change_prob = min(commit_count / working_days, 1.0) if working_days > 0 else 0.0
    r_cost = rebuild_cost(G, t, target_df)
    cost_rows.append({
        'cmake_target': t,
        'change_prob': change_prob,
        'rebuild_cost_ms': r_cost,
        'expected_daily_cost_ms': change_prob * r_cost,
    })
cost_df = pd.DataFrame(cost_rows).sort_values('expected_daily_cost_ms', ascending=False)
cost_lookup = cost_df.set_index('cmake_target')
total_daily_ms = cost_df['expected_daily_cost_ms'].sum()

group_map = dict(zip(fg_df['cmake_target'], fg_df['feature_group'])) if not fg_df.empty else {}
label_col = 'owning_group_label' if 'owning_group_label' in ownership_df.columns else 'owning_group_id'
ownership_map = dict(zip(ownership_df['cmake_target'], ownership_df[label_col])) if not ownership_df.empty else {}

print(f'Split recs:         {len(split_recs)} rows')
print(f'Optimised mapping:  {len(optimised_mapping)} rows')
print(f'Thin dependencies:  {len(thin_df)} rows')
print(f'Feature groups:     {len(fg_df)} rows')
print(f'Ownership:          {len(ownership_df)} rows')






## 2. Action Compilation

Compile all proposed actions from NB07–NB08: build performance, structural changes, dependency cleanup, ownership changes, schema changes.


In [ ]:
# ── Effort constants (engineer-days) ─────────────────────────────────────────
EFFORT = {
    'split': 5.0,
    'merge': 2.0,
    'extract': 3.0,
    'cleanup': 1.0,
    'schema': 3.0,
    'pch': 1.5,
    'iwyu': 3.0,
    'ownership': 1.0,
}

actions = []

# ═══════════════════════════════════════════════════════════════════════════════
# Source 1 — Target splits (NB07)
# ═══════════════════════════════════════════════════════════════════════════════
actionable_splits = split_recs[split_recs['recommended_k'] > 1].copy() if not split_recs.empty else pd.DataFrame()
for _, row in actionable_splits.iterrows():
    t = row['cmake_target']
    saving = int(row.get('compile_saving_ms') or 0)
    actions.append({
        'category': 'structural',
        'action_type': 'split',
        'cmake_target': t,
        'owning_team': ownership_map.get(t, 'unknown'),
        'build_time_impact_ms': saving,
        'modularity_impact': int(row.get('cross_partition_includes') or 0),
        'effort_days': EFFORT['split'],
        'risk_rating': risk_lookup.get(t, 'Low'),
        'on_cp': t in cp_set,
        'detail': (
            f"Split into {int(row.get('recommended_k', 2))} partitions "
            f"({row.get('method', 'spectral')}, balance={float(row.get('balance_ratio', 0)):.2f})"
        ),
        'depends_on': '',
    })

print(f'Split actions: {len(actionable_splits)}')

# ═══════════════════════════════════════════════════════════════════════════════
# Source 2 — Interface extractions (thin deps, NB07)
# ═══════════════════════════════════════════════════════════════════════════════
for _, row in thin_df.iterrows():
    src_t = row['depending_target']
    dst_t = row['depended_target']
    if dst_t not in G:
        continue
    dst_compile = float(G.nodes[dst_t].get('compile_time_sum_ms') or 0)
    # Saving: fraction of dep target's compile time we'd move to interface
    ratio = float(row.get('thinness_ratio') or 0)
    saving = int(dst_compile * max(0.0, 1 - ratio - 0.1))  # conservative
    actions.append({
        'category': 'structural',
        'action_type': 'extract',
        'cmake_target': f'{dst_t} → {dst_t}_iface',
        'owning_team': ownership_map.get(dst_t, 'unknown'),
        'build_time_impact_ms': saving,
        'modularity_impact': 1,
        'effort_days': EFFORT['extract'],
        'risk_rating': 'Low',
        'on_cp': dst_t in cp_set,
        'detail': f'Extract interface from {dst_t} (thinness={ratio:.3f})',
        'depends_on': '',
    })

print(f'Interface extraction actions: {sum(1 for a in actions if a["action_type"] == "extract")}')

# ═══════════════════════════════════════════════════════════════════════════════
# Source 3 — Dependency cleanup (unused deps)
# ═══════════════════════════════════════════════════════════════════════════════
import json
import os

target_dirs = defaultdict(set)
for _, row in file_df.iterrows():
    sf = row.get('source_file')
    if pd.isna(sf) or not sf:
        continue
    d = os.path.normpath(os.path.dirname(sf))
    target_dirs[row['cmake_target']].add(d)

target_includes = defaultdict(set)
for _, row in file_df.iterrows():
    ht = row.get('header_tree')
    if pd.isna(ht) if isinstance(ht, float) else not ht:
        continue
    try:
        tree = json.loads(ht) if isinstance(ht, str) else ht
    except (json.JSONDecodeError, TypeError):
        continue
    for entry in tree:
        if isinstance(entry, list) and len(entry) >= 2:
            target_includes[row['cmake_target']].add(os.path.normpath(str(entry[1])))

direct_link = edge_df[
    edge_df['is_direct'] & (edge_df['dependency_type'] == 'link')
] if 'dependency_type' in edge_df.columns else edge_df[edge_df['is_direct']]

for _, edge_row in direct_link.iterrows():
    src_t = edge_row['source_target']
    dst_t = edge_row['dest_target']
    dst_dirs = target_dirs.get(dst_t, set())
    if not dst_dirs:
        continue
    src_includes = target_includes.get(src_t, set())
    has_backing = any(
        any(inc.startswith(d) for d in dst_dirs) for inc in src_includes
    )
    if not has_backing:
        actions.append({
            'category': 'dependency_cleanup',
            'action_type': 'cleanup',
            'cmake_target': f'{src_t} → {dst_t}',
            'owning_team': ownership_map.get(src_t, 'unknown'),
            'build_time_impact_ms': 0,
            'modularity_impact': 1,
            'effort_days': EFFORT['cleanup'],
            'risk_rating': 'Low',
            'on_cp': src_t in cp_set or dst_t in cp_set,
            'detail': f'Remove unused link dependency: {src_t} → {dst_t}',
            'depends_on': '',
        })

print(f'Dependency cleanup actions: {sum(1 for a in actions if a["action_type"] == "cleanup")}')

# ═══════════════════════════════════════════════════════════════════════════════
# Source 4 — PCH candidates (high header depth, especially on CP)
# ═══════════════════════════════════════════════════════════════════════════════
PCH_DEPTH_THRESHOLD = 18
for _, row in target_df.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    hd = int(row.get('header_depth_max') or 0)
    compile_ms = int(row.get('compile_time_sum_ms') or 0)
    if hd >= PCH_DEPTH_THRESHOLD and compile_ms > 0:
        saving = int(compile_ms * 0.15)
        actions.append({
            'category': 'build_performance',
            'action_type': 'pch',
            'cmake_target': t,
            'owning_team': ownership_map.get(t, 'unknown'),
            'build_time_impact_ms': saving,
            'modularity_impact': 0,
            'effort_days': EFFORT['pch'],
            'risk_rating': risk_lookup.get(t, 'Low'),
            'on_cp': t in cp_set,
            'detail': f'Add PCH: header_depth={hd}, compile={compile_ms:,} ms (~15% saving)',
            'depends_on': '',
        })

print(f'PCH actions: {sum(1 for a in actions if a["action_type"] == "pch")}')

# ═══════════════════════════════════════════════════════════════════════════════
# Source 5 — IWYU (high expansion ratio on critical path)
# ═══════════════════════════════════════════════════════════════════════════════
if 'expansion_ratio_mean' in target_df.columns:
    er_threshold = target_df['expansion_ratio_mean'].quantile(0.75)
    for _, row in target_df.iterrows():
        t = row['cmake_target']
        if t not in cp_set or t not in G:
            continue
        er = float(row.get('expansion_ratio_mean') or 0)
        compile_ms = int(row.get('compile_time_sum_ms') or 0)
        if er > er_threshold and er > 0 and compile_ms > 0:
            saving = int(compile_ms * 0.20)
            actions.append({
                'category': 'build_performance',
                'action_type': 'iwyu',
                'cmake_target': t,
                'owning_team': ownership_map.get(t, 'unknown'),
                'build_time_impact_ms': saving,
                'modularity_impact': 0,
                'effort_days': EFFORT['iwyu'],
                'risk_rating': risk_lookup.get(t, 'Low'),
                'on_cp': True,
                'detail': f'IWYU cleanup: expansion_ratio={er:.0f}x (P75={er_threshold:.0f}x)',
                'depends_on': '',
            })

print(f'IWYU actions: {sum(1 for a in actions if a["action_type"] == "iwyu")}')

# ═══════════════════════════════════════════════════════════════════════════════
# Source 6 — Schema change isolation (codegen targets)
# ═══════════════════════════════════════════════════════════════════════════════
codegen_targets = target_df[target_df['codegen_file_count'].fillna(0) > 0].copy()
rev_G = G.reverse()
for _, row in codegen_targets.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    dependant_count = sum(
        1 for _ in nx.descendants(rev_G, t) if _ in G
    ) if t in rev_G else 0
    if dependant_count >= 3:
        codegen_ms = int(row.get('codegen_time_ms') or 0)
        actions.append({
            'category': 'schema',
            'action_type': 'schema',
            'cmake_target': t,
            'owning_team': ownership_map.get(t, 'unknown'),
            'build_time_impact_ms': codegen_ms,
            'modularity_impact': 0,
            'effort_days': EFFORT['schema'],
            'risk_rating': risk_lookup.get(t, 'Medium'),
            'on_cp': t in cp_set,
            'detail': (
                f'Isolate codegen behind interface library '
                f'({dependant_count} downstream targets, codegen={codegen_ms:,} ms)'
            ),
            'depends_on': '',
        })

print(f'Schema isolation actions: {sum(1 for a in actions if a["action_type"] == "schema")}')

# ═══════════════════════════════════════════════════════════════════════════════
# Source 7 — Ownership changes (contested targets)
# ═══════════════════════════════════════════════════════════════════════════════
if not ownership_df.empty and 'is_contested' in ownership_df.columns:
    contested = ownership_df[ownership_df['is_contested']].copy()
    for _, row in contested.iterrows():
        t = row['cmake_target']
        actions.append({
            'category': 'ownership',
            'action_type': 'ownership',
            'cmake_target': t,
            'owning_team': str(row.get('owning_group_label', 'unknown')),
            'build_time_impact_ms': 0,
            'modularity_impact': 0,
            'effort_days': EFFORT['ownership'],
            'risk_rating': 'Medium',
            'on_cp': t in cp_set,
            'detail': f'Clarify ownership (bus_factor={row.get("bus_factor", "?")})',
            'depends_on': '',
        })

    print(f'Ownership actions: {sum(1 for a in actions if a["action_type"] == "ownership")}')

# ═══════════════════════════════════════════════════════════════════════════════
# Assemble actions DataFrame
# ═══════════════════════════════════════════════════════════════════════════════
action_df = pd.DataFrame(actions)
action_df.insert(0, 'action_id', range(1, len(action_df) + 1))

print(f'\n{"=" * 60}')
print(f'Total actions compiled: {len(action_df)}')
print(f'{"=" * 60}')
if not action_df.empty:
    print(action_df.groupby('action_type')['action_id'].count().rename('count').to_string())
print()

## 3. Impact Scoring

For each action: build time impact (ms/day), modularity impact (cross-group edge reduction), effort estimate, risk rating from NB08.


In [ ]:
# ── Score each action ─────────────────────────────────────────────────────────
TYPE_COLORS = {
    'split': 'steelblue',
    'extract': '#8172B2',
    'cleanup': '#C44E52',
    'pch': '#CCB974',
    'iwyu': 'darkorange',
    'schema': '#DD8452',
    'ownership': '#64B5CD',
}

CONFIDENCE_BY_TYPE = {
    'split': 'High',
    'extract': 'Medium',
    'cleanup': 'Low',
    'pch': 'Low',
    'iwyu': 'Low',
    'schema': 'Medium',
    'ownership': 'Low',
}

scored_rows = []
for _, row in action_df.iterrows():
    atype = row['action_type']
    t_str = str(row['cmake_target'])
    total_saving_ms = int(row.get('build_time_impact_ms') or 0)

    # Look up change probability for the primary target
    primary_target = t_str.split(' → ')[0].split(',')[0].strip()
    if not cost_lookup.empty and primary_target in cost_lookup.index:
        change_prob = float(cost_lookup.loc[primary_target, 'change_prob'])
    else:
        change_prob = 0.01  # small default for structural changes

    # Daily saving
    if atype in ('split', 'pch', 'iwyu'):
        daily_ms = change_prob * total_saving_ms
    elif atype in ('extract', 'schema'):
        daily_ms = change_prob * total_saving_ms * 0.5  # more uncertain
    else:
        daily_ms = 0.0

    scored_rows.append({
        **row.to_dict(),
        'daily_saving_ms': round(daily_ms, 2),
        'confidence': CONFIDENCE_BY_TYPE.get(atype, 'Low'),
    })

scored_df = pd.DataFrame(scored_rows).sort_values('daily_saving_ms', ascending=False).reset_index(drop=True)

print('Scored actions (top 15):')
display(scored_df[['action_id', 'action_type', 'cmake_target', 'daily_saving_ms',
                    'build_time_impact_ms', 'effort_days', 'confidence',
                    'on_cp', 'risk_rating']].head(15).reset_index(drop=True))

# ── Two-panel plot ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Panel 1: top actions by daily saving
top15 = scored_df[scored_df['daily_saving_ms'] > 0].head(15)
if not top15.empty:
    colors = [TYPE_COLORS.get(t, 'grey') for t in top15['action_type']]
    ax1.barh(range(len(top15)), top15['daily_saving_ms'].values, color=colors)
    ax1.set_yticks(range(len(top15)))
    ax1.set_yticklabels(
        [f"[{r['action_type']}] {str(r['cmake_target'])[:28]}" for _, r in top15.iterrows()],
        fontsize=8,
    )
    ax1.invert_yaxis()
    ax1.set_xlabel('Expected daily saving (ms/day)')
    ax1.set_title('Top Actions by Daily Build Time Saving')
    legend_types = top15['action_type'].unique()
    ax1.legend(
        handles=[Patch(color=TYPE_COLORS.get(t, 'grey'), label=t) for t in legend_types],
        loc='lower right', fontsize=8,
    )
else:
    ax1.text(0.5, 0.5, 'No actions with positive daily saving', transform=ax1.transAxes, ha='center')

# Panel 2: effort vs total saving scatter
has_saving = scored_df[scored_df['build_time_impact_ms'] > 0].copy()
if not has_saving.empty:
    sizes = (has_saving['daily_saving_ms'] / (has_saving['daily_saving_ms'].max() or 1) * 300 + 30)
    colors2 = [TYPE_COLORS.get(t, 'grey') for t in has_saving['action_type']]
    ax2.scatter(has_saving['effort_days'], has_saving['build_time_impact_ms'],
                s=sizes.values, c=colors2, alpha=0.7, edgecolors='black', linewidths=0.5)
    for _, r in has_saving.head(12).iterrows():
        ax2.annotate(str(r['cmake_target'])[:18],
                     (r['effort_days'], r['build_time_impact_ms']),
                     fontsize=7, ha='left', va='bottom')
    ax2.set_xlabel('Effort (engineer-days)')
    ax2.set_ylabel('Total build time saving (ms)')
    ax2.set_title('Total Saving vs Effort (size = daily saving)')
else:
    ax2.text(0.5, 0.5, 'No actions with measurable savings', transform=ax2.transAxes, ha='center')

plt.tight_layout()
plt.show()

positive = scored_df[scored_df['daily_saving_ms'] > 0]
print(f'\nActions with positive daily saving: {len(positive)} / {len(scored_df)}')
print(f'Total potential daily saving: {positive["daily_saving_ms"].sum():,.1f} ms/day '
      f'({positive["daily_saving_ms"].sum() / 1000:.2f} s/day)')
print(f'Total effort required: {positive["effort_days"].sum():.1f} engineer-days')

## 4. Dependency Ordering

Build a dependency graph of actions (e.g. splits before reassignment, extractions before dep removal). Topological sort to show which can start immediately.


In [ ]:
# ── Build action dependency graph ─────────────────────────────────────────────
# Rules:
#  1. split(X) → cleanup(X → Y)  [split before removing cross-partition deps]
#  2. extract(X → X_iface) → cleanup(Y → X)  [extract before removing thick dep]
#  3. split(X) → schema(X)  [split before isolating codegen if same target]

G_actions = nx.DiGraph()
action_ids = list(scored_df['action_id'].values)
G_actions.add_nodes_from(action_ids)

# Index by target and type for fast lookups
by_target_type = defaultdict(list)
for _, row in scored_df.iterrows():
    by_target_type[(str(row['cmake_target']), row['action_type'])].append(row['action_id'])

by_target = defaultdict(list)
for _, row in scored_df.iterrows():
    by_target[str(row['cmake_target'])].append((row['action_type'], row['action_id']))

# Rule 1: splits must precede any cleanup on the same target
for _, split_row in scored_df[scored_df['action_type'] == 'split'].iterrows():
    split_t = str(split_row['cmake_target'])
    for atype, aid in by_target.get(split_t, []):
        if atype == 'cleanup' and aid != split_row['action_id']:
            G_actions.add_edge(split_row['action_id'], aid,
                                label=f'split {split_t} → cleanup')

# Rule 2: extractions must precede dependency cleanup on the destination target
for _, ext_row in scored_df[scored_df['action_type'] == 'extract'].iterrows():
    # extract target is 'X → X_iface'; base target is X
    ext_t_str = str(ext_row['cmake_target'])
    dst_t = ext_t_str.split(' → ')[0]
    # Any cleanup that removes a dependency ON dst_t
    for _, cl_row in scored_df[scored_df['action_type'] == 'cleanup'].iterrows():
        cl_t = str(cl_row['cmake_target'])
        if f'→ {dst_t}' in cl_t:
            G_actions.add_edge(ext_row['action_id'], cl_row['action_id'],
                                label=f'extract {dst_t} → cleanup dep on it')

# Rule 3: structural changes (splits/extracts) before ownership reassignment on same target
for _, struct_row in scored_df[scored_df['action_type'].isin(['split', 'extract'])].iterrows():
    base_t = str(struct_row['cmake_target']).split(' → ')[0]
    for atype, aid in by_target.get(base_t, []):
        if atype == 'ownership':
            G_actions.add_edge(struct_row['action_id'], aid,
                                label='structural change → ownership reassignment')

# ── Topological sort ──────────────────────────────────────────────────────────
try:
    topo_actions = list(nx.topological_sort(G_actions))
    cycle_free = True
except nx.NetworkXUnfeasible:
    topo_actions = action_ids
    cycle_free = False
    print('Warning: cycles in action dependency graph — showing original order.')

# Assign wave (0 = can start immediately)
wave = {aid: 0 for aid in action_ids}
for aid in topo_actions:
    preds = list(G_actions.predecessors(aid))
    if preds:
        wave[aid] = max(wave[p] for p in preds) + 1

scored_df['wave'] = scored_df['action_id'].map(wave)

print(f'Action dependency edges: {G_actions.number_of_edges()}')
print(f'Cycle-free: {cycle_free}')
print()
for w in sorted(scored_df['wave'].unique()):
    wave_actions = scored_df[scored_df['wave'] == w]
    print(f'Wave {w} — can start {"immediately" if w == 0 else "after wave " + str(w - 1)} '
          f'({len(wave_actions)} actions):')
    for _, r in wave_actions.iterrows():
        print(f'  #{r["action_id"]:3d} [{r["action_type"]:10s}] {str(r["cmake_target"])[:40]}')
    print()

## 5. ROI Ranking

ROI = (build_time_impact + modularity_impact_normalised) / effort_estimate. Top 10 overall and top 3 per team.


In [ ]:
# ── Compute ROI ───────────────────────────────────────────────────────────────
roi_df = scored_df.copy()

# Normalise modularity impact to [0, 1] range
max_mod = roi_df['modularity_impact'].max()
roi_df['modularity_impact_norm'] = (
    roi_df['modularity_impact'] / max(max_mod, 1)
)

# Normalise daily saving to [0, 1] range
max_daily = roi_df['daily_saving_ms'].max()
roi_df['daily_saving_norm'] = roi_df['daily_saving_ms'] / max(max_daily, 1)

# ROI: combined normalised impact / effort
roi_df['combined_score'] = roi_df['daily_saving_norm'] + 0.3 * roi_df['modularity_impact_norm']
roi_df['roi'] = roi_df['combined_score'] / roi_df['effort_days'].clip(lower=0.1)
roi_df = roi_df.sort_values('roi', ascending=False).reset_index(drop=True)
roi_df['rank'] = range(1, len(roi_df) + 1)

# Tier assignment
roi_positive = roi_df[roi_df['roi'] > 0]
roi_median = roi_positive['roi'].median() if not roi_positive.empty else 0


def assign_tier(row):
    if row['roi'] > roi_median and row['on_cp']:
        return 'Tier 1 — Critical Path Priority'
    elif row['roi'] > roi_median:
        return 'Tier 2 — High ROI'
    elif row['roi'] > 0:
        return 'Tier 3 — Standard'
    else:
        return 'Tier 4 — Structural / Maintenance'


roi_df['tier'] = roi_df.apply(assign_tier, axis=1)

TIER_COLORS = {
    'Tier 1 — Critical Path Priority': 'crimson',
    'Tier 2 — High ROI': 'darkorange',
    'Tier 3 — Standard': 'steelblue',
    'Tier 4 — Structural / Maintenance': '#9E9E9E',
}

# ── Top 10 overall ────────────────────────────────────────────────────────────
print('=== Top 10 Actions by ROI ===')
top10 = roi_df.head(10)
display(top10[['rank', 'action_type', 'cmake_target', 'roi', 'daily_saving_ms',
               'effort_days', 'confidence', 'tier', 'wave']].reset_index(drop=True))

# ── Top 3 per team ────────────────────────────────────────────────────────────
print('\n=== Top 3 Per Team ===')
for team in sorted(roi_df['owning_team'].unique()):
    if team in ('unknown', ''):
        continue
    team_actions = roi_df[roi_df['owning_team'] == team].head(3)
    print(f'\n  Team: {team}')
    for _, r in team_actions.iterrows():
        print(f'    #{r["rank"]:3d} [{r["action_type"]:10s}] {str(r["cmake_target"])[:35]:35s}'
              f'  ROI={r["roi"]:6.3f}  save={r["daily_saving_ms"]:7.1f} ms/day'
              f'  effort={r["effort_days"]:.1f}d')

# ── ROI visualisation ─────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Panel 1: ROI bar chart
roi_pos = roi_df[roi_df['roi'] > 0].head(20)
if not roi_pos.empty:
    colors = [TIER_COLORS[t] for t in roi_pos['tier']]
    ax1.barh(range(len(roi_pos)), roi_pos['roi'].values, color=colors)
    ax1.set_yticks(range(len(roi_pos)))
    ax1.set_yticklabels(
        [f"#{r['rank']} [{r['action_type']}] {str(r['cmake_target'])[:22]}"
         for _, r in roi_pos.iterrows()],
        fontsize=8,
    )
    ax1.invert_yaxis()
    if roi_median > 0:
        ax1.axvline(roi_median, color='black', linestyle='--', alpha=0.6,
                    label=f'Median ROI ({roi_median:.3f})')
    ax1.set_xlabel('ROI (combined score / effort day)')
    ax1.set_title('Top 20 Actions Ranked by ROI')
    ax1.legend(
        handles=[Patch(color=c, label=t) for t, c in TIER_COLORS.items()],
        loc='lower right', fontsize=7,
    )
else:
    ax1.text(0.5, 0.5, 'No actions with positive ROI', transform=ax1.transAxes, ha='center')

# Panel 2: ROI vs daily saving scatter by confidence
confidence_markers = {'High': 'o', 'Medium': 's', 'Low': 'D'}
for conf, marker in confidence_markers.items():
    sub = roi_pos[roi_pos['confidence'] == conf] if not roi_pos.empty else pd.DataFrame()
    if not sub.empty:
        colors_s = [TYPE_COLORS.get(t, 'grey') for t in sub['action_type']]
        ax2.scatter(sub['daily_saving_ms'], sub['roi'],
                    s=80, c=colors_s, marker=marker, alpha=0.7,
                    edgecolors='black', linewidths=0.5, label=f'{conf} confidence')
        for _, r in sub.iterrows():
            ax2.annotate(str(r['cmake_target'])[:15],
                         (r['daily_saving_ms'], r['roi']),
                         fontsize=7, ha='left', va='bottom')

ax2.set_xlabel('Daily saving (ms/day)')
ax2.set_ylabel('ROI')
ax2.set_title('ROI vs Daily Saving (shape = confidence)')
if not roi_pos.empty:
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Reporting

Generate `data/results/recommendations.csv` (full action list) and `data/results/executive_summary.md` (one-page summary).


In [ ]:
# ── Save full recommendations CSV ─────────────────────────────────────────────
out_cols = [
    'action_id', 'rank', 'tier', 'wave',
    'category', 'action_type', 'cmake_target', 'owning_team',
    'build_time_impact_ms', 'daily_saving_ms', 'modularity_impact',
    'effort_days', 'confidence', 'risk_rating', 'on_cp',
    'roi', 'detail', 'depends_on',
]
export_cols = [c for c in out_cols if c in roi_df.columns]
roi_df[export_cols].to_csv('../data/results/recommendations.csv', index=False)
print('Saved: data/results/recommendations.csv')

# ── Compute summary statistics ─────────────────────────────────────────────────
n_targets = G.number_of_nodes()
n_edges = G.number_of_edges()
n_groups = len(fg_df['feature_group'].unique()) if not fg_df.empty else 0
cp_len_s = cp_length / 1000
daily_cost_s = total_daily_ms / 1000

top5 = roi_df.head(5)
total_saving_top5 = top5['daily_saving_ms'].sum()
total_effort_top5 = top5['effort_days'].sum()

tier1_actions = roi_df[roi_df['tier'] == 'Tier 1 — Critical Path Priority']
tier2_actions = roi_df[roi_df['tier'] == 'Tier 2 — High ROI']

# Estimate total daily saving if all Tier 1+2 actions are done
high_value = roi_df[roi_df['tier'].isin(['Tier 1 — Critical Path Priority', 'Tier 2 — High ROI'])]
total_potential_saving_s = high_value['daily_saving_ms'].sum() / 1000
total_potential_effort = high_value['effort_days'].sum()

# Timeline estimate: 5 eng working in parallel on 8h days
PARALLEL_ENG = 5
timeline_days = total_potential_effort / PARALLEL_ENG

today = date.today().isoformat()

# ── Generate executive summary markdown ──────────────────────────────────────
summary_lines = [
    '# Build Optimisation Executive Summary',
    '',
    f'Generated: {today}',
    '',
    '## Current State',
    '',
    f'- **{n_targets}** CMake targets, **{n_edges}** dependency edges',
    f'- Critical path: **{cp_len_s:.1f} s**',
    f'- Expected incremental rebuild cost: **{daily_cost_s:.1f} s/day** (sum over all committers)',
    f'- Feature groups identified: **{n_groups}**',
    f'- Total actions identified: **{len(roi_df)}**',
    '',
    '## Proposed Structure',
    '',
    f'- Feature group partition enables per-team scoped builds.',
]

if not optimised_mapping.empty and 'scenario' in optimised_mapping.columns:
    current_m = optimised_mapping[optimised_mapping['scenario'] == 'current']
    proposed_m = optimised_mapping[optimised_mapping['scenario'] == 'proposed']
    if not current_m.empty and not proposed_m.empty:
        avg_current = current_m['feature_groups_required'].mean()
        avg_proposed = proposed_m['feature_groups_required'].mean()
        summary_lines.append(
            f'- Average feature groups per executable: **{avg_current:.1f}** → **{avg_proposed:.1f}** '
            f'(reduction of {avg_current - avg_proposed:.1f})'
        )

summary_lines += [
    '',
    '## Expected Improvement',
    '',
    f'- Implementing Tier 1 + Tier 2 actions ({len(high_value)} total):',
    f'  - Daily build time saving: **{total_potential_saving_s:.1f} s/day**',
    f'  - Total effort: **{total_potential_effort:.0f} engineer-days**',
    f'  - Estimated timeline at {PARALLEL_ENG} engineers: **{timeline_days:.0f} working days**',
    '',
    '## Top 5 Actions',
    '',
]

for i, (_, r) in enumerate(top5.iterrows(), 1):
    cp_flag = ' *(critical path)*' if r['on_cp'] else ''
    summary_lines.append(
        f'{i}. **[{r["action_type"]}]** `{str(r["cmake_target"])[:50]}` — '
        f'{r["detail"][:80]}  '
        f'(*{r["confidence"]} confidence, {r["effort_days"]:.0f}d effort, '
        f'~{r["daily_saving_ms"]:.0f} ms/day saving{cp_flag}*)'
    )

summary_lines += [
    '',
    '## Estimated Timeline',
    '',
    '| Wave | Actions | Focus |',
    '|------|---------|-------|',
]

for w in sorted(roi_df['wave'].unique()):
    wave_rows = roi_df[roi_df['wave'] == w]
    focus_types = ', '.join(wave_rows['action_type'].value_counts().head(3).index.tolist())
    summary_lines.append(f'| Wave {w} | {len(wave_rows)} | {focus_types} |')

summary_lines += [
    '',
    '---',
    f'*Generated by build-optimiser notebooks. Full details in `data/results/recommendations.csv`.*',
]

summary_text = '\n'.join(summary_lines)

summary_path = results_dir / 'executive_summary.md'
summary_path.write_text(summary_text, encoding='utf-8')
print(f'Saved: data/results/executive_summary.md')
print()
print(summary_text)